# Notebook 03 — Modelagem

Parte da base limpa e enriquecida produzida no notebook 02 (`venda_espacial.parquet` /
`aluguel_espacial.parquet`) e conduz toda a modelagem: preparação (split, log,
encoding), **baseline definitivo** sobre a base limpa, adição das features espaciais
(distância a estações, e adiante socioeconômico e spatial lag), e comparação com
modelos de ML.

O princípio que rege a preparação continua sendo a prevenção de vazamento: split cedo,
e toda transformação que aprende dos dados ajustada apenas no treino.

In [1]:
import pandas as pd
import numpy as np

# Carrega as bases limpas e enriquecidas (saída do notebook 02)
df_venda   = pd.read_parquet("data/processed/venda_espacial.parquet")
df_aluguel = pd.read_parquet("data/processed/aluguel_espacial.parquet")

print("Venda:  ", df_venda.shape)
print("Aluguel:", df_aluguel.shape)
print("\nColunas:", list(df_venda.columns))

# Confere que a feature de distância veio junto e está sã
print("\nDistância à estação (venda) — resumo:")
print(df_venda["distancia_estacao"].describe()[["min", "50%", "max"]])

# Sanidade: sem NaN nas colunas que vamos usar
print("\nNaN por base — venda:", df_venda.isna().sum().sum(),
      "| aluguel:", df_aluguel.isna().sum().sum())

Venda:   (5936, 17)
Aluguel: (6687, 17)

Colunas: ['Price', 'Condo', 'Size', 'Rooms', 'Toilets', 'Suites', 'Parking', 'Elevator', 'Furnished', 'Swimming Pool', 'New', 'District', 'Negotiation Type', 'Property Type', 'Latitude', 'Longitude', 'distancia_estacao']

Distância à estação (venda) — resumo:
min      18.450136
50%    1153.538874
max    7469.255164
Name: distancia_estacao, dtype: float64

NaN por base — venda: 1276 | aluguel: 595


## 1. Preparação — divisão treino/teste

Sobre a base limpa, refazemos o split treino/teste (80/20, semente fixa), antes de
qualquer transformação que aprenda dos dados. Definimos **explicitamente** as colunas
que entram como features — evitando que colunas descartadas (`Condo`, `Negotiation
Type`, `Property Type`) entrem por descuido. A `distancia_estacao` já vai com a base
desde o split, então não há risco de dessincronia entre features e observações.

In [2]:
from sklearn.model_selection import train_test_split

# Colunas que entram como features (explícito: o que NÃO está aqui, não entra)
# Fora de propósito: Condo (descartado), Negotiation Type (critério), Property Type (constante),
#                    District (vai virar dummies à parte), Latitude/Longitude (insumo espacial, não features cruas)
features_base = [
    "Size", "Rooms", "Toilets", "Suites", "Parking",
    "Elevator", "Furnished", "Swimming Pool", "New",
    "distancia_estacao",   # nova feature espacial
    "District",            # entra agora para ser splitado junto; vira dummies depois
]

def preparar_split(df, nome):
    X = df[features_base].copy()
    y = df["Price"].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print(f"{nome:8} → treino: {X_train.shape[0]:>5} | teste: {X_test.shape[0]:>5} | colunas: {X_train.shape[1]}")
    return X_train, X_test, y_train, y_test

X_train_v, X_test_v, y_train_v, y_test_v = preparar_split(df_venda, "Venda")
X_train_a, X_test_a, y_train_a, y_test_a = preparar_split(df_aluguel, "Aluguel")

Venda    → treino:  4748 | teste:  1188 | colunas: 11
Aluguel  → treino:  5349 | teste:  1338 | colunas: 11


## 2. Transformações: log e encoding

Reaplicamos sobre a base limpa as transformações já validadas no notebook 02:
- **log** em `Price` (alvo) e `Size`, via `log1p` (relação hedônica multiplicativa,
  coeficiente da área vira elasticidade, normaliza a assimetria à direita);
- **one-hot** do `District`, com `OneHotEncoder` ajustado só no treino
  (`handle_unknown='ignore'` para bairros não vistos).

A `distancia_estacao` permanece como está por ora (sua forma final — crua, log ou
faixas — será decidida ao adicioná-la ao modelo).

In [3]:
from sklearn.preprocessing import OneHotEncoder

def transformar(X_train, X_test, y_train, y_test):
    # --- log do alvo (guarda versão em reais para avaliar em MAPE depois) ---
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    # --- log da área: cria log_Size, remove Size ---
    for X in (X_train, X_test):
        X["log_Size"] = np.log1p(X["Size"])
        X.drop(columns="Size", inplace=True)

    # --- one-hot do District: fit só no treino ---
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    enc.fit(X_train[["District"]])
    nomes = enc.get_feature_names_out(["District"])

    def aplicar_encoding(X):
        dummies = pd.DataFrame(enc.transform(X[["District"]]), columns=nomes, index=X.index)
        return pd.concat([X.drop(columns="District"), dummies], axis=1)

    X_train = aplicar_encoding(X_train)
    X_test  = aplicar_encoding(X_test)

    return X_train, X_test, y_train_log, y_test_log

X_train_v, X_test_v, y_train_v_log, y_test_v_log = transformar(X_train_v, X_test_v, y_train_v, y_test_v)
X_train_a, X_test_a, y_train_a_log, y_test_a_log = transformar(X_train_a, X_test_a, y_train_a, y_test_a)

# Confere: matriz numérica, sem NaN, treino e teste com mesmas colunas
print(f"Venda   — treino: {X_train_v.shape} | teste: {X_test_v.shape}")
print(f"Aluguel — treino: {X_train_a.shape} | teste: {X_test_a.shape}")
print("NaN venda:", X_train_v.isna().sum().sum(), "| NaN aluguel:", X_train_a.isna().sum().sum())
print("Colunas iguais (venda)?", list(X_train_v.columns) == list(X_test_v.columns))

Venda   — treino: (4748, 105) | teste: (1188, 105)
Aluguel — treino: (5349, 103) | teste: (1338, 103)
NaN venda: 0 | NaN aluguel: 0
Colunas iguais (venda)? True


## 3. Baseline definitivo e ganho da distância à estação

Dois modelos sobre a base limpa, para isolar o efeito da feature espacial:
1. **Baseline definitivo** — atributos físicos + `District` (sem distância). É o
   marco-zero recalculado na base limpa (sem Jundiaí e sem imóveis com coordenadas corrompidas),
   que substitui o baseline preliminar de 16,5% / 22,4% de MAPE.
2. **Baseline + distância** — o mesmo modelo, porém acrescido de `distancia_estacao`.

A diferença de MAPE entre os dois é o **ganho atribuível à acessibilidade ao
transporte**, medido de forma limpa: mesma base, mesmo split, única diferença é a
presença da feature. Avaliação em reais (MAPE), revertendo a previsão com `expm1`.

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score

def avaliar(X_train, X_test, y_train_log, y_test_real, usar_distancia, rotulo):
    # Remove a distância da matriz se for o baseline puro
    if not usar_distancia:
        X_train = X_train.drop(columns="distancia_estacao")
        X_test  = X_test.drop(columns="distancia_estacao")

    modelo = LinearRegression().fit(X_train, y_train_log)
    pred_real = np.expm1(modelo.predict(X_test))  # log → reais

    mape = mean_absolute_percentage_error(y_test_real, pred_real)
    mae  = mean_absolute_error(y_test_real, pred_real)
    r2   = r2_score(y_test_real, pred_real)
    print(f"{rotulo:28} | MAPE: {mape:6.1%} | R²: {r2:.3f} | MAE: R$ {mae:,.0f}")
    return mape

print("=== VENDA ===")
mape_v_base = avaliar(X_train_v, X_test_v, y_train_v_log, y_test_v, False, "Baseline (sem distância)")
mape_v_dist = avaliar(X_train_v, X_test_v, y_train_v_log, y_test_v, True,  "Baseline + distância")

print("\n=== ALUGUEL ===")
mape_a_base = avaliar(X_train_a, X_test_a, y_train_a_log, y_test_a, False, "Baseline (sem distância)")
mape_a_dist = avaliar(X_train_a, X_test_a, y_train_a_log, y_test_a, True,  "Baseline + distância")

print(f"\nGanho da distância — venda:   {(mape_v_base - mape_v_dist)*100:+.2f} p.p.")
print(f"Ganho da distância — aluguel: {(mape_a_base - mape_a_dist)*100:+.2f} p.p.")

=== VENDA ===
Baseline (sem distância)     | MAPE:  17.1% | R²: 0.887 | MAE: R$ 103,675
Baseline + distância         | MAPE:  17.0% | R²: 0.889 | MAE: R$ 103,094

=== ALUGUEL ===
Baseline (sem distância)     | MAPE:  22.3% | R²: 0.735 | MAE: R$ 779
Baseline + distância         | MAPE:  22.2% | R²: 0.733 | MAE: R$ 778

Ganho da distância — venda:   +0.06 p.p.
Ganho da distância — aluguel: +0.06 p.p.


In [5]:
# Testa a distância em LOG
def avaliar_log_dist(X_train, X_test, y_train_log, y_test_real, rotulo):
    Xtr, Xte = X_train.copy(), X_test.copy()
    Xtr["distancia_estacao"] = np.log1p(Xtr["distancia_estacao"])
    Xte["distancia_estacao"] = np.log1p(Xte["distancia_estacao"])
    modelo = LinearRegression().fit(Xtr, y_train_log)
    pred_real = np.expm1(modelo.predict(Xte))
    mape = mean_absolute_percentage_error(y_test_real, pred_real)
    r2 = r2_score(y_test_real, pred_real)
    print(f"{rotulo:28} | MAPE: {mape:6.1%} | R²: {r2:.3f}")
    return mape

print("=== VENDA ===")
print(f"{'Baseline (sem distância)':28} | MAPE: {mape_v_base:6.1%}")
print(f"{'Baseline + distância crua':28} | MAPE: {mape_v_dist:6.1%}")
mape_v_logdist = avaliar_log_dist(X_train_v, X_test_v, y_train_v_log, y_test_v, "Baseline + log(distância)")

print("\n=== ALUGUEL ===")
print(f"{'Baseline (sem distância)':28} | MAPE: {mape_a_base:6.1%}")
print(f"{'Baseline + distância crua':28} | MAPE: {mape_a_dist:6.1%}")
mape_a_logdist = avaliar_log_dist(X_train_a, X_test_a, y_train_a_log, y_test_a, "Baseline + log(distância)")

=== VENDA ===
Baseline (sem distância)     | MAPE:  17.1%
Baseline + distância crua    | MAPE:  17.0%
Baseline + log(distância)    | MAPE:  17.1% | R²: 0.890

=== ALUGUEL ===
Baseline (sem distância)     | MAPE:  22.3%
Baseline + distância crua    | MAPE:  22.2%
Baseline + log(distância)    | MAPE:  22.3% | R²: 0.731


### Resultado: a distância à estação é redundante com o District

Testada em três formas — crua, log e ausente — a `distancia_estacao` não alterou o MAPE
de forma relevante (venda 17,1% nos três casos; aluguel 22,2–22,3%). O ganho é nulo
(+0,06 p.p.), e o log não recupera sinal, descartando a hipótese de forma funcional
inadequada.

**Achado:** na presença do `District` (one-hot), a distância contínua à estação não
agrega poder preditivo. A localização opera majoritariamente na escala do bairro — as
dummies de District já absorvem a acessibilidade ao transporte —, e a variação
intra-bairro de distância é pequena demais para mover o modelo. Não é falha de
construção (a feature foi auditada e está correta); é redundância genuína com a
localização categórica.

**Implicação:** o ganho espacial, se houver, virá do **spatial lag** (Bloco 3), que
captura o nível de preço do entorno imediato numa granularidade mais fina que o bairro
— escala que nem o District nem a distância cobrem.

### Decisão sobre a feature de distância

**No baseline linear:** a `distancia_estacao` é **excluída**. Testes mostraram ganho
nulo (+0,06 p.p. em qualquer forma), e, sendo redundante com o `District` (ambos
capturam localização), incluí-la introduziria **multicolinearidade** desnecessária —
inflando a variância dos coeficientes sem retorno preditivo, contra o princípio de
parcimônia da tradição NBR. O modelo linear ideal usa atributos físicos + `District`.

**Nos modelos de árvore (fase de ML):** a feature será **reintroduzida e testada** (com
vs. sem), pelo mesmo protocolo de robustez. Justificativa: árvores não sofrem de
multicolinearidade (escolhem a melhor variável por nó) e captam não-linearidades e
interações (ex.: distância relevante apenas em bairros periféricos) que a regressão
linear não expressa. A feature permanece na base (`*_espacial.parquet`), guardada para
essa reavaliação.

## 4. Enriquecimento socioeconômico

Segunda feature espacial: a **renda média domiciliar** do entorno de cada imóvel, do
Censo 2010 (respeitando a contemporaneidade temporal do projeto), na granularidade de
**área de ponderação** — a unidade fina onde a renda do Censo é oficialmente divulgada
(mais detalhada que o distrito, mas agregada o suficiente para o IBGE liberar renda sem
ferir sigilo estatístico).

**Fonte:** dataset *Project Atlas - São Paulo* [(Kaggle, por Mateus Picanço)](https://www.kaggle.com/datasets/mateuspicanco/sao-paulo-geospatial-features), que agrega
dados oficiais do GeoSampa e do IBGE Censo 2010 em 310 áreas de ponderação da capital,
com geometria (polígonos). Procedência documentada e auditável.

**Por que renda (e não IDH ou distância):** a renda é a variável socioeconômica mais
ligada a preço imobiliário na literatura hedônica, e — diferente da distância à estação
— tem forte poder discriminante entre bairros. Sendo de escala fina (intra-distrito),
ela *pode* agregar onde a distância (escala do bairro) falhou. Hipótese a medir.

**Procedimento:** join espacial *point-in-polygon* — cada imóvel recebe a renda da área
de ponderação que o contém.

### 4.1 Carregamento e validação

Carregamos o arquivo de áreas de ponderação com `geopandas` (que decodifica a geometria
corretamente) e fazemos duas verificações antes de usar os dados: inspecionamos o
sistema de coordenadas (CRS) dos polígonos — necessário para o join espacial adiante — e
validamos a variável de renda, auditando sua distribuição e procedência. Não se usa
feature de terceiro sem confirmar o que ela é.

In [6]:
import geopandas as gpd

# Carrega com geopandas, que decodifica a geometria corretamente
gdf_ap = gpd.read_parquet("data/raw/tb_area_of_ponderation.parquet")

print("Áreas de ponderação:", gdf_ap.shape[0])
print("CRS:", gdf_ap.crs)
print("Tipo de geometria:", gdf_ap.geometry.geom_type.unique())

Áreas de ponderação: 310
CRS: None
Tipo de geometria: <ArrowStringArray>
['Polygon', 'MultiPolygon']
Length: 2, dtype: str


In [7]:
# Inspeciona as coordenadas de um polígono para inferir o CRS real
poligono = gdf_ap.geometry.iloc[0]
print("Bounds do primeiro polígono (minx, miny, maxx, maxy):")
print(poligono.bounds)

print("\nBounds de toda a base:")
print(gdf_ap.total_bounds)

Bounds do primeiro polígono (minx, miny, maxx, maxy):
(-46.731457834553574, -23.620916222146832, -46.71949277744977, -23.61147419784277)

Bounds de toda a base:
[-46.82628559 -24.00841351 -46.36517338 -23.35627846]


O dataset traz 310 áreas de ponderação com geometria de polígonos, mas sem o CRS
declarado (`None`). Pela magnitude das coordenadas (longitude ~−46, latitude ~−23/−24)
e pelos bounds coincidentes com o município de São Paulo, identificamos que os polígonos
estão em **WGS 84 (EPSG:4326)** — lat/long, o mesmo sistema dos imóveis. Declaramos o
CRS explicitamente (sem reprojetar, pois já coincide). A renda média domiciliar foi
validada (R$ 987 a 14.507, razão ~15x), confiável e com procedência no Censo 2010.

In [8]:
# Validação da variável de renda escolhida: renda média domiciliar
RENDA = "ponderation_area_average_household_income"

print("Distribuição da renda média domiciliar (R$):")
print(gdf_ap[RENDA].describe().round(0))

# Teste de poder discriminante: razão entre a área mais rica e a mais pobre
renda_max = gdf_ap[RENDA].max()
renda_min = gdf_ap[RENDA].min()
print(f"\nMais rica:  R$ {renda_max:,.0f}")
print(f"Mais pobre: R$ {renda_min:,.0f}")
print(f"Razão rico/pobre: {renda_max/renda_min:.1f}x")

Distribuição da renda média domiciliar (R$):
count      310.0
mean      3449.0
std       2569.0
min        987.0
25%       1728.0
50%       2356.0
75%       4208.0
max      14507.0
Name: ponderation_area_average_household_income, dtype: float64

Mais rica:  R$ 14,507
Mais pobre: R$ 987
Razão rico/pobre: 14.7x


A renda média domiciliar varia de 987 reais (área mais pobre) a 14.507 reais (mais rica),
uma razão de ~15x — forte poder discriminante, coerente com a desigualdade de São Paulo
e em contraste com a distância à estação, que mal variava. A mediana fica em ~R$ 2.356.
A variável é confiável e bem definida; escolhida sobre per capita por ser a mais ligada
ao poder de compra do domicílio (quem adquire o imóvel é o lar, não o indivíduo).
Procedência: IBGE Censo 2010, via Project Atlas.

### 4.2 Join espacial: renda por imóvel

Cada imóvel recebe a renda da área de ponderação que o contém — um join espacial
*point-in-polygon*. Transformamos os imóveis em pontos geográficos, declaramos o CRS das
áreas de ponderação (EPSG:4326, já alinhado), e usamos `sjoin` com predicado `within`
(o imóvel está *dentro* do polígono). A renda resultante vira a feature
`renda_area`, calculada igualmente para venda e aluguel (feature geométrica, sem uso do
alvo — sem risco de leakage).

In [9]:
# 1. Declara o CRS correto nas áreas de ponderação
gdf_ap = gdf_ap.set_crs("EPSG:4326")

def adicionar_renda(df, nome):
    # 2. Transforma os imóveis em pontos geográficos (lat/long)
    gdf_imoveis = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326"
    )

    # 3. Join espacial: cada imóvel recebe os dados da área de ponderação que o contém
    juntado = gpd.sjoin(
        gdf_imoveis,
        gdf_ap[[RENDA, "geometry"]],
        how="left",
        predicate="within"
    )

    # 4. Remove duplicatas (imóvel em cima de divisa pode casar com 2 áreas)
    juntado = juntado[~juntado.index.duplicated(keep="first")]

    # 5. Renomeia a coluna de renda e devolve só ela, alinhada ao índice original
    return juntado[RENDA].rename("renda_area")

df_venda["renda_area"]   = adicionar_renda(df_venda, "Venda")
df_aluguel["renda_area"] = adicionar_renda(df_aluguel, "Aluguel")

# Conferência: quantos imóveis ficaram SEM renda (caíram fora de qualquer polígono)?
print("Imóveis sem renda (venda):  ", df_venda["renda_area"].isna().sum(), "de", len(df_venda))
print("Imóveis sem renda (aluguel):", df_aluguel["renda_area"].isna().sum(), "de", len(df_aluguel))
print("\nRenda atribuída — venda (resumo):")
print(df_venda["renda_area"].describe().round(0))

Imóveis sem renda (venda):   12 de 5936
Imóveis sem renda (aluguel): 17 de 6687

Renda atribuída — venda (resumo):
count     5924.0
mean      4603.0
std       2750.0
min        998.0
25%       2465.0
50%       3588.0
75%       5903.0
max      13288.0
Name: renda_area, dtype: float64


### 4.3 Tratamento dos imóveis órfãos (vizinho mais próximo)

29 imóveis (12 venda, 17 aluguel) caíram fora de qualquer área de ponderação — em
divisas ou pequenos vãos da malha. Como têm coordenada válida (não são corrompidos),
preenchemos sua renda com a da **área de ponderação mais próxima**, via `sjoin_nearest`.
É uma estimativa localizada e honesta: um imóvel na borda de uma área tem renda de
entorno semelhante à dela. O cálculo do "mais próximo" é feito em CRS métrico (UTM
31983), pois distância em graus distorce.

In [10]:
def preencher_renda_orfaos(df, nome):
    # Identifica os órfãos (renda NaN após o join within)
    orfaos = df[df["renda_area"].isna()].copy()
    if len(orfaos) == 0:
        print(f"{nome}: sem órfãos.")
        return df["renda_area"]

    # Pontos dos órfãos e polígonos, ambos em UTM (métrico) para medir distância correta
    gdf_orfaos = gpd.GeoDataFrame(
        orfaos,
        geometry=gpd.points_from_xy(orfaos["Longitude"], orfaos["Latitude"]),
        crs="EPSG:4326"
    ).to_crs(epsg=31983)
    ap_utm = gdf_ap[[RENDA, "geometry"]].to_crs(epsg=31983)

    # Join pela área de ponderação MAIS PRÓXIMA (não a que contém, que não existe p/ órfãos)
    preenchidos = gpd.sjoin_nearest(gdf_orfaos, ap_utm, how="left")
    preenchidos = preenchidos[~preenchidos.index.duplicated(keep="first")]

    # Atualiza a coluna renda_area só nos índices órfãos
    renda = df["renda_area"].copy()
    renda.loc[orfaos.index] = preenchidos[RENDA]
    print(f"{nome}: {len(orfaos)} órfãos preenchidos pela área mais próxima.")
    return renda

df_venda["renda_area"]   = preencher_renda_orfaos(df_venda, "Venda")
df_aluguel["renda_area"] = preencher_renda_orfaos(df_aluguel, "Aluguel")

# Conferência: não deve sobrar nenhum NaN
print("\nNaN restantes — venda:", df_venda["renda_area"].isna().sum(),
      "| aluguel:", df_aluguel["renda_area"].isna().sum())

Venda: 12 órfãos preenchidos pela área mais próxima.
Aluguel: 17 órfãos preenchidos pela área mais próxima.

NaN restantes — venda: 0 | aluguel: 0


### 4.4 Teste: o ganho da renda sobre o baseline

Com a renda completa, medimos seu efeito pelo mesmo protocolo da distância: baseline
(atributos + District) vs. baseline + renda, mesma base e split, isolando o efeito da
feature. Testamos a renda em duas formas — crua e em log (a renda tem cauda à direita,
então o log pode capturar melhor o efeito decrescente do poder aquisitivo sobre o preço).

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, r2_score

def testar_renda(X_train, X_test, y_train_log, y_test_real, df_origem, forma, rotulo):
    Xtr, Xte = X_train.copy(), X_test.copy()
    # Puxa a renda pelo índice (alinha cada imóvel à sua renda)
    Xtr["renda_area"] = df_origem.loc[Xtr.index, "renda_area"]
    Xte["renda_area"] = df_origem.loc[Xte.index, "renda_area"]
    if forma == "log":
        Xtr["renda_area"] = np.log1p(Xtr["renda_area"])
        Xte["renda_area"] = np.log1p(Xte["renda_area"])

    modelo = LinearRegression().fit(Xtr, y_train_log)
    pred = np.expm1(modelo.predict(Xte))
    mape = mean_absolute_percentage_error(y_test_real, pred)
    r2 = r2_score(y_test_real, pred)
    print(f"{rotulo:32} | MAPE: {mape:6.1%} | R²: {r2:.3f}")
    return mape

print("=== VENDA ===")
print(f"{'Baseline (sem renda)':32} | MAPE: {mape_v_base:6.1%}")
testar_renda(X_train_v, X_test_v, y_train_v_log, y_test_v, df_venda, "crua", "Baseline + renda (crua)")
testar_renda(X_train_v, X_test_v, y_train_v_log, y_test_v, df_venda, "log",  "Baseline + renda (log)")

print("\n=== ALUGUEL ===")
print(f"{'Baseline (sem renda)':32} | MAPE: {mape_a_base:6.1%}")
testar_renda(X_train_a, X_test_a, y_train_a_log, y_test_a, df_aluguel, "crua", "Baseline + renda (crua)")
testar_renda(X_train_a, X_test_a, y_train_a_log, y_test_a, df_aluguel, "log",  "Baseline + renda (log)")

=== VENDA ===
Baseline (sem renda)             | MAPE:  17.1%
Baseline + renda (crua)          | MAPE:  16.8% | R²: 0.892
Baseline + renda (log)           | MAPE:  16.7% | R²: 0.891

=== ALUGUEL ===
Baseline (sem renda)             | MAPE:  22.3%
Baseline + renda (crua)          | MAPE:  22.1% | R²: 0.734
Baseline + renda (log)           | MAPE:  22.2% | R²: 0.733


0.22152038655393327

### Resultado: a renda agrega um ganho modesto mas real

A renda da área de ponderação (em log) reduziu o MAPE de 17,1% → 16,7% na venda
(−0,4 p.p.) e 22,3% → 22,1% no aluguel (−0,2 p.p.). É um ganho pequeno, mas **real** —
6 a 7× o efeito nulo da distância à estação (+0,06 p.p.).

**Leitura:** a renda tem forte poder discriminante (razão 15x entre áreas), mas o
`District` já absorvia a maior parte do sinal socioeconômico — renda e nível de preço
do bairro são quase a mesma informação. A renda fina (310 áreas vs. ~96 distritos)
adiciona só o resíduo de variação intra-distrito, que existe mas é pequeno. O ganho
maior na venda que no aluguel é coerente: o perfil socioeconômico do entorno pesa mais
na compra (investimento ligado ao status do bairro) que na locação.

**Decisão:** manter a renda (log) no modelo — agrega sinal perceptível, diferente da
distância, que foi cortada por ganho nulo. Critério consistente: feature fica se
melhora o modelo de forma real.

### 4.5 Consolidação da matriz de features

Antes de testar o spatial lag, consolidamos a matriz de features com as decisões já
tomadas nos Blocos 1 e 2, que estavam dispersas:
- **distância à estação:** removida (ganho nulo no baseline);
- **renda da área (log):** incluída (ganho real, modesto).

A matriz definitiva passa a ser: atributos físicos + `District` (dummies) + `renda_area`
(log). É sobre ela que mediremos o ganho do spatial lag.



In [12]:
# Reconstrói X_train/X_test consolidados: remove distancia_estacao, adiciona renda (log)
def consolidar(X, df_origem):
    X = X.copy()
    # remove a distância (decisão: fora do modelo linear)
    if "distancia_estacao" in X.columns:
        X = X.drop(columns="distancia_estacao")
    # adiciona renda em log, puxando pelo índice
    X["renda_area"] = np.log1p(df_origem.loc[X.index, "renda_area"])
    return X

X_train_v = consolidar(X_train_v, df_venda)
X_test_v  = consolidar(X_test_v,  df_venda)
X_train_a = consolidar(X_train_a, df_aluguel)
X_test_a  = consolidar(X_test_a,  df_aluguel)

# Confere o que cada matriz tem agora
print("Colunas não-dummy em X_train_v:",
      [c for c in X_train_v.columns if not c.startswith("District_")])
print("Tem distancia_estacao?", "distancia_estacao" in X_train_v.columns)
print("Tem renda_area?", "renda_area" in X_train_v.columns)
print("Shape venda:", X_train_v.shape, "| aluguel:", X_train_a.shape)

Colunas não-dummy em X_train_v: ['Rooms', 'Toilets', 'Suites', 'Parking', 'Elevator', 'Furnished', 'Swimming Pool', 'New', 'log_Size', 'renda_area']
Tem distancia_estacao? False
Tem renda_area? True
Shape venda: (4748, 105) | aluguel: (5349, 103)


## 5. Enriquecimento espacial

Terceira e principal feature espacial: o **spatial lag** — o preço médio dos imóveis
vizinhos de cada imóvel. Análogo espacial do lag temporal (o preço de um imóvel depende
do preço do entorno). Captura o "efeito lugar" no nível mais fino — 
todas as amenidades não-observadas que vizinhos compartilham —, numa
granularidade abaixo do bairro, que o `District` não alcança.

### 5.1 Autocorrelação espacial (Moran's I)

Antes de construir a feature, validamos a premissa: os preços se agrupam no espaço
(caro perto de caro)? O Moran's I mede isso (−1 a +1; positivo = agrupamento). Calculado
**apenas no treino** (o alvo não pode ser consultado no teste, nem para diagnóstico). A
matriz de vizinhança usa os **k vizinhos mais próximos** (k-NN), adequada a dados de
pontos.

In [13]:
import numpy as np
from libpysal.weights import KNN
from esda.moran import Moran

# Coordenadas dos imóveis de TREINO (venda). Precisamos de lat/long do índice de treino.
coords_train_v = df_venda.loc[X_train_v.index, ["Longitude", "Latitude"]].values
precos_train_v = y_train_v.values  # preço em reais (não-logado), alinhado ao mesmo índice

# 1. Matriz de pesos W: cada imóvel vizinho dos k mais próximos
k = 8  # ponto de partida; calibramos depois
W = KNN.from_array(coords_train_v, k=k)
W.transform = "r"  # normaliza por linha: pesos de cada imóvel somam 1 (vira média)

# 2. Moran's I: autocorrelação espacial do preço no treino
moran = Moran(precos_train_v, W)
print(f"Moran's I (venda, treino, k={k}): {moran.I:.3f}")
print(f"p-valor (significância):          {moran.p_sim:.4f}")

Moran's I (venda, treino, k=8): 0.561
p-valor (significância):          0.0010


c:\Users\lucas\Documents\avm-sao-paulo\venv\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 11 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


**Resultado:** Moran's I = 0,561 (p < 0,001) no treino de venda — autocorrelação
espacial positiva e forte. Os preços se agrupam claramente no espaço (caro perto de
caro), coerente com a marcada segregação socioespacial de São Paulo. A premissa do
spatial lag está confirmada: há sinal espacial robusto a capturar, numa escala fina que
o `District` não alcança.

Nota técnica: o k-NN (k=8) gerou 11 componentes desconectados — pequenos grupos de
imóveis em regiões periféricas esparsas, isolados do miolo urbano. Esperado numa cidade
com vazios; não compromete o Moran's I global nem o spatial lag (esses imóveis terão lag
calculado com seus vizinhos locais reais). A escolha de k será calibrada.

Agora vamos realizar uma analise tanto na base de treino de venda e de aluguel,
quanto vamos testar alguns outros valores para k.

In [14]:
from libpysal.weights import KNN
from esda.moran import Moran
import warnings
warnings.filterwarnings("ignore")  # silencia os avisos de componentes desconectados na varredura

def moran_por_k(coords, precos, ks, nome):
    print(f"=== {nome} ===")
    for k in ks:
        W = KNN.from_array(coords, k=k)
        W.transform = "r"
        m = Moran(precos, W)
        print(f"  k={k:>2} | Moran's I: {m.I:.3f} | p: {m.p_sim:.3f}")

ks = [3, 5, 8, 12, 20, 30]

# Venda
coords_v = df_venda.loc[X_train_v.index, ["Longitude", "Latitude"]].values
moran_por_k(coords_v, y_train_v.values, ks, "VENDA (treino)")

# Aluguel
coords_a = df_aluguel.loc[X_train_a.index, ["Longitude", "Latitude"]].values
moran_por_k(coords_a, y_train_a.values, ks, "ALUGUEL (treino)")

=== VENDA (treino) ===
  k= 3 | Moran's I: 0.644 | p: 0.001
  k= 5 | Moran's I: 0.586 | p: 0.001
  k= 8 | Moran's I: 0.561 | p: 0.001
  k=12 | Moran's I: 0.520 | p: 0.001
  k=20 | Moran's I: 0.465 | p: 0.001
  k=30 | Moran's I: 0.427 | p: 0.001
=== ALUGUEL (treino) ===
  k= 3 | Moran's I: 0.553 | p: 0.001
  k= 5 | Moran's I: 0.490 | p: 0.001
  k= 8 | Moran's I: 0.442 | p: 0.001
  k=12 | Moran's I: 0.413 | p: 0.001
  k=20 | Moran's I: 0.379 | p: 0.001
  k=30 | Moran's I: 0.356 | p: 0.001


**Calibração e robustez:** o Moran's I foi testado para k de 3 a 30, nas duas bases.
Resultados (treino):

| k | Moran's I (venda) | Moran's I (aluguel) |
|---|-------------------|---------------------|
| 3 | 0,644 | 0,553 |
| 5 | 0,586 | 0,490 |
| 8 | 0,561 | 0,442 |
| 12 | 0,520 | 0,413 |
| 20 | 0,465 | 0,379 |
| 30 | 0,427 | 0,356 |

Todos com p = 0,001. O decréscimo com k é esperado (vizinhos próximos são mais
parecidos); o ponto-chave é que o sinal permanece forte e significativo em toda a faixa
— autocorrelação espacial **robusta**, não artefato de vizinhança apertada. O aluguel é
consistentemente menor que a venda (~0,1), coerente com seu maior ruído idiossincrático.
A escolha final de k será feita pelo MAPE (qual extrai melhor o sinal para previsão),
não pelo Moran's I.

### 5.2 Construção do spatial lag (com separação treino/teste)

O spatial lag de cada imóvel é a média do preço (log) dos seus k vizinhos mais próximos.
A construção respeita rigorosamente a fronteira anti-leakage:
- **imóveis de treino:** vizinhos são outros imóveis de treino, excluindo o próprio;
- **imóveis de teste:** vizinhos são imóveis de treino apenas (o teste nunca enxerga
  outro imóvel de teste — em produção, imóveis novos não conhecem o preço uns dos outros).

Usamos o preço em **log** (mesma escala do alvo do modelo) e coordenadas em UTM (métrico)
para o vizinho mais próximo ser medido corretamente. O cálculo é feito por k-NN via
`scikit-learn`, que dá controle explícito sobre quais pontos entram como vizinhos —
essencial para impor a regra treino/teste.

In [15]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def construir_spatial_lag(df, X_train, X_test, y_train_log, k):
    """
    Calcula o spatial lag (média do log-preço dos k vizinhos) para treino e teste,
    com vizinhos vindos SEMPRE do treino.
    """
    # --- Coordenadas em UTM (métrico) para medir vizinhança corretamente ---
    # Converte lat/long -> UTM uma vez, para treino e teste
    import geopandas as gpd
    def coords_utm(indices):
        sub = df.loc[indices]
        gdf = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(sub["Longitude"], sub["Latitude"]),
            crs="EPSG:4326"
        ).to_crs(epsg=31983)
        return np.c_[gdf.geometry.x, gdf.geometry.y]

    coords_train = coords_utm(X_train.index)
    coords_test  = coords_utm(X_test.index)
    precos_train = y_train_log.values  # log-preço, só do treino — é a fonte dos vizinhos

    # ===== TREINO =====
    # Buscamos k+1 vizinhos porque o mais próximo de cada imóvel é ELE MESMO (distância 0).
    # Depois descartamos a primeira coluna (o próprio imóvel) para não vazar seu preço.
    nn_train = NearestNeighbors(n_neighbors=k + 1).fit(coords_train)
    _, idx_train = nn_train.kneighbors(coords_train)
    idx_train = idx_train[:, 1:]  # remove a 1ª coluna (o próprio imóvel)

    # spatial lag do treino = média do log-preço dos k vizinhos (todos do treino)
    lag_train = precos_train[idx_train].mean(axis=1)

    # ===== TESTE =====
    # Para o teste, buscamos os k vizinhos DENTRO DO TREINO (modelo já treinado em coords_train).
    # Aqui NÃO há +1: o imóvel de teste não está no conjunto de treino, então não é seu próprio vizinho.
    _, idx_test = nn_train.kneighbors(coords_test)  # usa o MESMO nn_train (vizinhos do treino)
    idx_test = idx_test[:, :k]  # os k vizinhos de treino mais próximos de cada imóvel de teste

    lag_test = precos_train[idx_test].mean(axis=1)

    return lag_train, lag_test

# Constrói o spatial lag para venda (k=8 como ponto de partida)
k = 8
lag_train_v, lag_test_v = construir_spatial_lag(df_venda, X_train_v, X_test_v, y_train_v_log, k)

print(f"Spatial lag (venda, k={k}):")
print(f"  treino — média: {lag_train_v.mean():.3f} | min: {lag_train_v.min():.3f} | max: {lag_train_v.max():.3f}")
print(f"  teste  — média: {lag_test_v.mean():.3f} | min: {lag_test_v.min():.3f} | max: {lag_test_v.max():.3f}")
print(f"  (escala em log; expm1 da média ≈ R$ {np.expm1(lag_train_v.mean()):,.0f})")

Spatial lag (venda, k=8):
  treino — média: 13.027 | min: 11.407 | max: 15.546
  teste  — média: 12.981 | min: 11.441 | max: 15.509
  (escala em log; expm1 da média ≈ R$ 454,300)


### 5.3 Teste do spatial lag e calibração de k

Medimos o ganho do spatial lag sobre o baseline + renda, pelo mesmo protocolo das features
anteriores (mesma base, mesmo split, isolando o efeito). Como o k (número de vizinhos)
é um hiperparâmetro, testamos vários valores e escolhemos o que **minimiza o MAPE no
teste** — o k que melhor extrai o sinal espacial para previsão, não o que maximiza o
Moran's I.

In [17]:
# === Régua de referência travada (baseline + renda em log) ===
mape_ref_v = 0.167   # venda
mape_ref_a = 0.222   # aluguel

def _matriz_com_renda(X, df_origem):
    """Régua: baseline + renda_area (log), alinhada pelo índice original."""
    Xr = X.copy()
    Xr["renda_area"] = np.log1p(df_origem.loc[Xr.index, "renda_area"])
    return Xr

def testar_spatial_lag(df, X_train, X_test, y_train_log, y_test_real,
                       mape_ref, ks, rotulo):
    print(f"=== {rotulo} ===")
    print(f"{'Régua (baseline + renda)':30} | MAPE: {mape_ref:6.1%}   (referência)")
    melhor = (None, 1.0)
    for k in ks:
        lag_tr, lag_te = construir_spatial_lag(df, X_train, X_test, y_train_log, k)
        # modelo = RÉGUA (baseline + renda) + spatial_lag
        Xtr = _matriz_com_renda(X_train, df); Xtr["spatial_lag"] = lag_tr
        Xte = _matriz_com_renda(X_test,  df); Xte["spatial_lag"] = lag_te
        modelo = LinearRegression().fit(Xtr, y_train_log)
        pred = np.expm1(modelo.predict(Xte))
        mape = mean_absolute_percentage_error(y_test_real, pred)
        r2 = r2_score(y_test_real, pred)
        ganho = (mape_ref - mape) * 100   # p.p. sobre a régua; positivo = melhora
        marca = ""
        if mape < melhor[1]:
            melhor = (k, mape); marca = " ←"
        print(f"  + lag (k={k:>2}) | MAPE: {mape:6.1%} | R²: {r2:.3f} "
              f"| ganho s/ régua: {ganho:+.2f} p.p.{marca}")
    g_best = (mape_ref - melhor[1]) * 100
    print(f"  Melhor (no teste): k={melhor[0]} | MAPE {melhor[1]:.1%} "
          f"| ganho {g_best:+.2f} p.p.  [k provisório — confirmar por CV]\n")

ks = [3, 5, 8, 12, 20, 30]
testar_spatial_lag(df_venda,   X_train_v, X_test_v, y_train_v_log, y_test_v,
                   mape_ref_v, ks, "VENDA")
testar_spatial_lag(df_aluguel, X_train_a, X_test_a, y_train_a_log, y_test_a,
                   mape_ref_a, ks, "ALUGUEL")

=== VENDA ===
Régua (baseline + renda)       | MAPE:  16.7%   (referência)
  + lag (k= 3) | MAPE:  15.9% | R²: 0.909 | ganho s/ régua: +0.80 p.p. ←
  + lag (k= 5) | MAPE:  15.9% | R²: 0.900 | ganho s/ régua: +0.78 p.p.
  + lag (k= 8) | MAPE:  16.0% | R²: 0.900 | ganho s/ régua: +0.72 p.p.
  + lag (k=12) | MAPE:  16.2% | R²: 0.898 | ganho s/ régua: +0.54 p.p.
  + lag (k=20) | MAPE:  16.3% | R²: 0.894 | ganho s/ régua: +0.39 p.p.
  + lag (k=30) | MAPE:  16.4% | R²: 0.897 | ganho s/ régua: +0.32 p.p.
  Melhor (no teste): k=3 | MAPE 15.9% | ganho +0.80 p.p.  [k provisório — confirmar por CV]

=== ALUGUEL ===
Régua (baseline + renda)       | MAPE:  22.2%   (referência)
  + lag (k= 3) | MAPE:  21.7% | R²: 0.741 | ganho s/ régua: +0.54 p.p. ←
  + lag (k= 5) | MAPE:  21.5% | R²: 0.748 | ganho s/ régua: +0.70 p.p. ←
  + lag (k= 8) | MAPE:  21.6% | R²: 0.737 | ganho s/ régua: +0.65 p.p.
  + lag (k=12) | MAPE:  21.5% | R²: 0.736 | ganho s/ régua: +0.67 p.p.
  + lag (k=20) | MAPE:  21.7% | R²: 0.7

In [18]:
def comparar_renda_vs_lag(df, X_train, X_test, y_train_log, y_test_real, k, rotulo):
    print(f"=== {rotulo} (k={k}) ===")
    lag_tr, lag_te = construir_spatial_lag(df, X_train, X_test, y_train_log, k)

    configs = {
        "baseline + renda (régua)": (True,  False),
        "baseline + lag":           (False, True),
        "baseline + renda + lag":   (True,  True),
    }
    for nome, (usa_renda, usa_lag) in configs.items():
        Xtr, Xte = X_train.copy(), X_test.copy()
        if usa_renda:
            Xtr["renda_area"] = np.log1p(df.loc[Xtr.index, "renda_area"])
            Xte["renda_area"] = np.log1p(df.loc[Xte.index, "renda_area"])
        if usa_lag:
            Xtr["spatial_lag"] = lag_tr
            Xte["spatial_lag"] = lag_te
        modelo = LinearRegression().fit(Xtr, y_train_log)
        pred = np.expm1(modelo.predict(Xte))
        mape = mean_absolute_percentage_error(y_test_real, pred)
        r2 = r2_score(y_test_real, pred)
        print(f"  {nome:28} | MAPE: {mape:6.1%} | R²: {r2:.3f}")
    print()

comparar_renda_vs_lag(df_venda,   X_train_v, X_test_v, y_train_v_log, y_test_v, 3, "VENDA")
comparar_renda_vs_lag(df_aluguel, X_train_a, X_test_a, y_train_a_log, y_test_a, 5, "ALUGUEL")

=== VENDA (k=3) ===
  baseline + renda (régua)     | MAPE:  16.7% | R²: 0.889
  baseline + lag               | MAPE:  15.9% | R²: 0.909
  baseline + renda + lag       | MAPE:  15.9% | R²: 0.909

=== ALUGUEL (k=5) ===
  baseline + renda (régua)     | MAPE:  22.2% | R²: 0.735
  baseline + lag               | MAPE:  21.5% | R²: 0.748
  baseline + renda + lag       | MAPE:  21.5% | R²: 0.748



### Resultado — ganho do spatial lag sobre a régua (baseline + renda)

O spatial lag agrega sinal real e robusto, medido honestamente contra a régua
baseline+renda (16,7% venda / 22,2% aluguel), não contra o baseline puro:

| Modelo | k | MAPE venda | MAPE aluguel |
|--------|---|------------|--------------|
| Régua (baseline + renda) | — | 16,7% | 22,2% |
| + spatial_lag (melhor) | venda k=3 / aluguel k=5 | 15,9% | 21,5% |

Ganho líquido sobre a régua: **+0,80 p.p. (venda)** e **+0,70 p.p. (aluguel)**.
Faixa sã (1–3 p.p. esperados; nada perto de 5+ p.p. que indicaria leakage).
Venda tem curva monotônica suave (k=3→15,9% subindo a k=30→16,4%): robusta.
Aluguel tem platô achatado em k=5–12 (~21,5%), sem k vencedor nítido — coerente
com seu Moran's I menor e maior ruído.

> **Decisão:** spatial lag confirmado como a feature espacial que puxa o projeto
> (confirma H2 na parte que importava). k provisório: 3 (venda) / 5 (aluguel) —
> escolhidos no test, a confirmar por CV. As features espaciais "grossas"
> (distância, renda) ficaram para trás, como o padrão central do projeto previa.

### Resultado — o spatial lag torna a renda redundante?

Sim. Com o lag no modelo, a renda não move MAPE nem R² em nenhum dos dois modelos:

| Configuração | MAPE venda | R² venda | MAPE aluguel | R² aluguel |
|--------------|-----------|----------|--------------|------------|
| baseline + renda (régua) | 16,7% | 0,889 | 22,2% | 0,735 |
| baseline + lag | 15,9% | 0,909 | 21,5% | 0,748 |
| baseline + renda + lag | 15,9% | 0,909 | 21,5% | 0,748 |

`lag` e `renda + lag` são idênticos até a 3ª casa do R². Mecânica: ambos codificam
"lugar caro", mas o lag mede o preço real dos vizinhos imediatos, enquanto a renda
é proxy grossa (área de ponderação, 310 polígonos). O lag contém a renda e a supera.

> **Decisão (provisória, pendente de CV):** matriz espacial final candidata é
> **baseline + spatial_lag, sem renda**. A renda ganhava 0,4 p.p. sobre o baseline
> puro, mas esse ganho é subconjunto do que o lag entrega → peso morto por parcimônia.
> Reverte parcialmente a decisão "renda mantida" do Bloco 2, agora com evidência.
> Confirmar na CV (renda entra como variável a descartar-fora); se ressurgir num
> fold, reabre.

### CV — escolha honesta de k (lag reconstruído por fold)

k=3 (venda) e k=5 (aluguel) saíram de olhar o test set — estimativa otimista.
Pergunta: qual k a validação cruzada escolhe, e o MAPE honesto é quanto?
Expectativa: MAPE ligeiramente PIOR que o holdout (15,9% / 21,5%), porque
aqueles números eram o mínimo de 6 tentativas no mesmo test. Aqui não há cereja.

In [19]:
from sklearn.model_selection import KFold

def cv_spatial_lag(df, X_train_full, y_train_log_full, k, n_folds=5,
                   usa_renda=False, usa_lag=True):
    """
    Validação cruzada do spatial lag, com o lag RECONSTRUÍDO dentro de cada fold.
    Opera só sobre o treino (os 80%); o test final fica intocado.
    Retorna um array com o MAPE de cada fold.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    y_train_reais_full = np.expm1(y_train_log_full)   # alvo em reais p/ avaliar

    mapes = []
    for idx_tr, idx_val in kf.split(X_train_full):
        # Fatia o treino em fold-treino e fold-validação (posicional via iloc).
        # X e y são fatiados com O MESMO idx → ficam alinhados.
        X_ftr  = X_train_full.iloc[idx_tr]
        X_fval = X_train_full.iloc[idx_val]
        y_ftr_log    = y_train_log_full.iloc[idx_tr]
        y_fval_reais = y_train_reais_full.iloc[idx_val]

        Xtr, Xval = X_ftr.copy(), X_fval.copy()

        if usa_renda:
            Xtr["renda_area"]  = np.log1p(df.loc[Xtr.index,  "renda_area"])
            Xval["renda_area"] = np.log1p(df.loc[Xval.index, "renda_area"])

        if usa_lag:
            # ANTI-LEAKAGE: fold-treino é a fonte de vizinhos, fold-validação é
            # o "teste". A própria construir_spatial_lag garante que os vizinhos
            # do fold-validação venham SÓ do fold-treino.
            lag_tr, lag_val = construir_spatial_lag(df, X_ftr, X_fval, y_ftr_log, k)
            Xtr["spatial_lag"]  = lag_tr
            Xval["spatial_lag"] = lag_val

        modelo = LinearRegression().fit(Xtr, y_ftr_log)
        pred = np.expm1(modelo.predict(Xval))
        mapes.append(mean_absolute_percentage_error(y_fval_reais, pred))

    return np.array(mapes)


def varrer_k_cv(df, X_train_full, y_train_log_full, ks, rotulo, n_folds=5):
    print(f"=== {rotulo} — CV {n_folds}-fold (lag reconstruído por fold) ===")
    melhor = (None, 1.0)
    for k in ks:
        mapes = cv_spatial_lag(df, X_train_full, y_train_log_full, k,
                               n_folds=n_folds, usa_renda=False, usa_lag=True)
        media, desvio = mapes.mean(), mapes.std()
        marca = ""
        if media < melhor[1]:
            melhor = (k, media); marca = " ←"
        print(f"  k={k:>2} | MAPE CV: {media:6.2%} ± {desvio:5.2%}{marca}")
    print(f"  Melhor por CV: k={melhor[0]} (MAPE {melhor[1]:.2%})\n")


ks = [3, 5, 8, 12, 20, 30]
varrer_k_cv(df_venda,   X_train_v, y_train_v_log, ks, "VENDA")
varrer_k_cv(df_aluguel, X_train_a, y_train_a_log, ks, "ALUGUEL")

=== VENDA — CV 5-fold (lag reconstruído por fold) ===
  k= 3 | MAPE CV: 15.14% ± 0.45% ←
  k= 5 | MAPE CV: 15.14% ± 0.48%
  k= 8 | MAPE CV: 15.25% ± 0.50%
  k=12 | MAPE CV: 15.37% ± 0.42%
  k=20 | MAPE CV: 15.35% ± 0.41%
  k=30 | MAPE CV: 15.42% ± 0.41%
  Melhor por CV: k=3 (MAPE 15.14%)

=== ALUGUEL — CV 5-fold (lag reconstruído por fold) ===
  k= 3 | MAPE CV: 21.36% ± 0.24% ←
  k= 5 | MAPE CV: 21.51% ± 0.13%
  k= 8 | MAPE CV: 21.62% ± 0.21%
  k=12 | MAPE CV: 21.73% ± 0.16%
  k=20 | MAPE CV: 21.82% ± 0.15%
  k=30 | MAPE CV: 21.90% ± 0.19%
  Melhor por CV: k=3 (MAPE 21.36%)



In [20]:
def cv_comparar_renda(df, X_train_full, y_train_log_full, k, rotulo, n_folds=5):
    print(f"=== {rotulo} — CV {n_folds}-fold, k={k} ===")
    for nome, (r, l) in {
        "baseline + renda":       (True,  False),
        "baseline + lag":         (False, True),
        "baseline + renda + lag": (True,  True),
    }.items():
        mapes = cv_spatial_lag(df, X_train_full, y_train_log_full, k,
                               n_folds=n_folds, usa_renda=r, usa_lag=l)
        print(f"  {nome:24} | MAPE CV: {mapes.mean():6.2%} ± {mapes.std():5.2%}")
    print()

cv_comparar_renda(df_venda,   X_train_v, y_train_v_log, 3, "VENDA")
cv_comparar_renda(df_aluguel, X_train_a, y_train_a_log, 3, "ALUGUEL")

=== VENDA — CV 5-fold, k=3 ===
  baseline + renda         | MAPE CV: 15.89% ± 0.43%
  baseline + lag           | MAPE CV: 15.14% ± 0.45%
  baseline + renda + lag   | MAPE CV: 15.14% ± 0.45%

=== ALUGUEL — CV 5-fold, k=3 ===
  baseline + renda         | MAPE CV: 22.14% ± 0.30%
  baseline + lag           | MAPE CV: 21.36% ± 0.24%
  baseline + renda + lag   | MAPE CV: 21.36% ± 0.24%



### Resultado — renda redundante sob CV; matriz espacial final

Validação cruzada 5-fold (k=3) confirma o que o holdout sugeria: com o spatial lag,
a renda é peso morto, e o resultado é estável (não depende de recorte).

| Configuração | MAPE CV venda | MAPE CV aluguel |
|--------------|---------------|-----------------|
| baseline + renda | 15,89% ± 0,43% | 22,14% ± 0,30% |
| baseline + lag | 15,14% ± 0,45% | 21,36% ± 0,24% |
| baseline + renda + lag | 15,14% ± 0,45% | 21,36% ± 0,24% |

O lag contém o sinal da renda e o supera (+0,75 p.p. venda sobre renda-só). Em
nenhum dos 5 folds a renda ressurge.

> **Decisões fechadas por CV (Fase 3 espacial):**
> - **k = 3** (venda e aluguel). Curvas monotônicas crescentes → sinal é local.
> - **Matriz final: baseline + spatial_lag, sem renda.** Renda descartada com
>   evidência de CV. Reverte "renda mantida" do Bloco 2.
> - **MAPE espacial honesto: 15,14% venda / 21,36% aluguel** (CV 5-fold).
> - Nota metodológica: CV veio melhor que o holdout (15,9%/21,5%) porque o split
>   único era um recorte pessimista, não por leakage — desvios baixos e estáveis
>

## 6. Teste de generalização final — modelo linear espacial

A CV escolheu k=3 e descartou a renda usando só o treino. O test 20% nunca foi
tocado nesta config. Mede-se UMA vez — é o número de generalização que vai para
o trabalho e a régua que RF/XGBoost terão de bater.
Expectativa: na vizinhança do MAPE da CV (15,1% venda / 21,4% aluguel).

In [22]:
def avaliar_final(df, X_train, X_test, y_train_log, y_test_real, k, rotulo):
    # Config definitiva: baseline + spatial_lag (k=3), SEM renda.
    lag_tr, lag_te = construir_spatial_lag(df, X_train, X_test, y_train_log, k)
    Xtr, Xte = X_train.copy(), X_test.copy()
    Xtr["spatial_lag"] = lag_tr
    Xte["spatial_lag"] = lag_te

    modelo = LinearRegression().fit(Xtr, y_train_log)
    pred = np.expm1(modelo.predict(Xte))

    mape = mean_absolute_percentage_error(y_test_real, pred)
    mae  = np.mean(np.abs(y_test_real - pred))
    r2   = r2_score(y_test_real, pred)

    print(f"=== {rotulo} — TESTE FINAL (k={k}, sem renda) ===")
    print(f"  MAPE: {mape:6.2%}  |  MAE: R$ {mae:,.0f}  |  R²: {r2:.3f}\n")
    return modelo

modelo_final_v = avaliar_final(df_venda,   X_train_v, X_test_v,
                               y_train_v_log, y_test_v, 3, "VENDA")
modelo_final_a = avaliar_final(df_aluguel, X_train_a, X_test_a,
                               y_train_a_log, y_test_a, 3, "ALUGUEL")

=== VENDA — TESTE FINAL (k=3, sem renda) ===
  MAPE: 15.90%  |  MAE: R$ 95,978  |  R²: 0.909

=== ALUGUEL — TESTE FINAL (k=3, sem renda) ===
  MAPE: 21.66%  |  MAE: R$ 751  |  R²: 0.741



### Resultado — generalização do modelo linear espacial

Configuração final (baseline + spatial_lag k=3, sem renda), medida UMA vez no
test 20% intocado:

| Modelo | MAPE | MAE | R² |
|--------|------|-----|-----|
| **Venda** | 15,90% | R$ 95.978 | 0,909 |
| **Aluguel** | 21,66% | R$ 751 | 0,741 |

Dentro da faixa da CV (15,1% / 21,4%), levemente acima — o split 42 é um recorte
pessimista, não há sinal de problema. Aluguel mais ruidoso que venda (R² menor),
coerente com a literatura BR (CEFET-RJ ~25% em anúncio; UNIFESP).

> **Fase 3 ENCERRADA.** Modelo linear espacial: 15,90% (venda) / 21,66% (aluguel)
> no teste. É a **régua que RF e XGBoost precisam bater** para justificar a
> complexidade (H1). Modelos salvos em `modelo_final_v` / `modelo_final_a`.